# Exploratory Data Analysis and Bayesian Popularity Recommender

**Portfolio notebook · Python, Pandas, NumPy, and Matplotlib**

## Project summary

This notebook examines an Amazon product-review dataset and develops a transparent, non-personalized recommendation baseline. The work begins with data quality checks and exploratory analysis, then measures user–item sparsity, explores rating and activity distributions, and ranks products with a Bayesian-smoothed popularity score.

The objective is not to claim a fully personalized recommendation engine. Instead, it establishes a reliable baseline that remains useful for new users, sparse interaction histories, and as a benchmark for later collaborative or hybrid models.

### What this notebook demonstrates

- Reproducible loading and cleaning of line-delimited review data
- Exploratory analysis of ratings, activity concentration, sparsity, and time
- A Bayesian popularity-ranking method that reduces small-sample bias
- A time-aware baseline evaluation approach and an honest discussion of limitations

## 1. Background Theory

### 1.1 The Recommendation Problem

A recommendation system can be formally defined as follows. Let $U = \{u_1, u_2, \ldots, u_m\}$ be the set of users and $I = \{i_1, i_2, \ldots, i_n\}$ be the set of items. The goal is to learn a utility function:

$$f: U \times I \rightarrow \mathbb{R}$$

that estimates how much a user $u$ would prefer item $i$, and then surface the top-$N$ items with the highest predicted utility for each user.

In practice, $f$ is unknown and must be inferred from observed interactions — ratings, clicks, purchases, or dwell time.

### 1.2 The User-Item Rating Matrix

The central data structure in collaborative recommendation is the **user-item matrix** $R \in \mathbb{R}^{m \times n}$, where $R_{ui}$ is the rating user $u$ gave item $i$. Most entries are missing because users interact with only a tiny fraction of the catalog. The fraction of observed entries is called the **density**:

$$\text{density} = \frac{|\{(u, i) : R_{ui} \text{ is observed}\}|}{m \times n}$$

Typical real-world datasets have densities below 1%. This extreme sparsity is one of the central challenges in recommendation system design.

### 1.3 The Cold-Start Problem

The cold-start problem arises when the system lacks sufficient interaction history to make reliable personalized recommendations. It manifests in two ways:

- **User cold-start:** A new user has no prior interactions, so their preferences are unknown.
- **Item cold-start:** A newly listed item has received no ratings, so it cannot surface through collaborative signals.

Popularity-based recommenders sidestep the cold-start problem entirely by relying only on aggregate statistics rather than individual history. This makes them a critical component of production systems as a fallback strategy.

### 1.4 Naive Popularity vs. Bayesian Average

Ranking items by their raw average rating is misleading. An item with 2 reviews averaging 5.0 stars should not outrank an item with 10,000 reviews averaging 4.7 stars.

The **Bayesian average** (also called the regularized mean) corrects for this by shrinking item estimates toward the global mean when the review count is low:

$$\hat{\mu}_i = \frac{v_i \cdot \bar{r}_i + C \cdot \mu}{v_i + C}$$

Where:
- $\bar{r}_i$ = mean rating of item $i$
- $v_i$ = number of ratings for item $i$
- $\mu$ = global mean rating across all items
- $C$ = smoothing constant (typically set to the mean or median review count across items)

As $v_i \to \infty$, the Bayesian average converges to the item's true mean. As $v_i \to 0$, it regresses to the global mean $\mu$.

## 1. Dataset and reproducible loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

### Expected data format

The analysis expects an Amazon Reviews file in JSON-lines format. The dataset itself is intentionally not included in this repository because it can be large and may be subject to separate distribution terms. Place the downloaded file in the local `data/` folder before running the notebook.

Relevant fields are:

| Field | Description |
|---|---|
| `reviewerID` | Anonymous user identifier |
| `asin` | Product identifier |
| `overall` | Numeric rating from 1.0 to 5.0 |
| `unixReviewTime` | Unix timestamp of the review |

In [ ]:
from pathlib import Path

# The dataset is intentionally not committed to the repository.
# Download a compatible Amazon Reviews JSON/JSON.GZ file and place it here.
DATA_PATH = Path("data/Electronics_5.json.gz")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Add a compatible Amazon Reviews JSON-lines file "
        "to the local data/ folder before running this notebook."
    )

chunks = []
for chunk in pd.read_json(DATA_PATH, lines=True, compression="infer", chunksize=50_000):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"Raw dataset shape: {df.shape}")
df.head()

## 2. Data preparation

In [ ]:
# Rename key columns for clarity
df = df.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'item_id',
    'overall': 'rating'
})

# Drop rows missing required fields
df = df.dropna(subset=['user_id', 'item_id', 'rating'])

# Convert timestamp to datetime
df['review_date'] = pd.to_datetime(df['unixReviewTime'], unit='s', errors='coerce')

# Remove rows with invalid parsed dates
df = df.dropna(subset=['review_date'])

# Keep the most recent review for each (user, item) pair
df = df.sort_values('review_date').drop_duplicates(subset=['user_id', 'item_id'], keep='last')

print(f"Dataset shape after cleaning: {df.shape}")
df.dtypes

## 3. Exploratory analysis

### 3.1 Rating distribution

Understanding the rating distribution is important because review data often contains positivity bias: users may be more likely to submit a rating after an unusually positive or negative experience. This means observed ratings are not necessarily a random sample of all product experiences.

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(rating_counts.index, rating_counts.values)
plt.xlabel('Rating Value')
plt.ylabel('Count')
plt.title('Rating Distribution')
plt.xticks([1, 2, 3, 4, 5])
plt.show()


### 3.2 User and item activity

Review activity commonly follows a long-tail pattern: a small number of users contribute many reviews, while a small number of products receive most ratings. This concentration affects both the fairness and the coverage of a recommender, especially for less popular items.

In [ ]:
# Compute per-user and per-item review counts
user_counts = df.groupby('user_id')['rating'].count().sort_values(ascending=False)
item_counts = df.groupby('item_id')['rating'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_counts, bins=50)
axes[0].set_title('User Activity Distribution')
axes[0].set_xlabel('Reviews per User')
axes[0].set_ylabel('Frequency')
axes[0].set_xscale('log')
axes[0].set_yscale('log')

axes[1].hist(item_counts, bins=50)
axes[1].set_title('Item Popularity Distribution')
axes[1].set_xlabel('Reviews per Item')
axes[1].set_ylabel('Frequency')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()


### 3.3 User–item sparsity

The observed density of the user–item matrix indicates how much behavioural overlap is available for personalized methods. Very low density makes nearest-neighbour recommendations unstable because many users have not rated the same items.

In [ ]:
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()
n_ratings = len(df)

density = n_ratings / (n_users * n_items)
sparsity = 1 - density

print(f"Users:    {n_users:,}")
print(f"Items:    {n_items:,}")
print(f"Ratings:  {n_ratings:,}")
print(f"Density:  {density:.6%}")
print(f"Sparsity: {sparsity:.6%}")

# High sparsity means user-user overlap is very low, so user-based collaborative
# filtering will struggle to find reliable neighbors and tends to be unstable.


### 3.4 Review activity over time

A temporal view of reviews helps identify changes in activity and motivates a chronological train/test split. This avoids leaking future interactions into a model trained to represent an earlier point in time.

In [ ]:
reviews_per_month = df.groupby(df['review_date'].dt.to_period('M')).size()
reviews_per_month.index = reviews_per_month.index.to_timestamp()

plt.figure(figsize=(12, 5))
plt.plot(reviews_per_month.index, reviews_per_month.values)
plt.title('Number of Reviews per Month')
plt.xlabel('Month')
plt.ylabel('Number of Reviews')
plt.tight_layout()
plt.show()


## 4. Bayesian popularity recommender

### 4.1 Regularized item scores

Raw mean ratings are unreliable for lightly reviewed products. The Bayesian average below incorporates the number of ratings and shrinks uncertain estimates toward the global mean.

In [ ]:
# Aggregate statistics per item
item_stats = df.groupby('item_id').agg(
    mean_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

# Global mean
mu = df['rating'].mean()

# Smoothing constant: median review count
C = item_stats['num_ratings'].median()

# Bayesian average score
item_stats['bayes_score'] = (
    (item_stats['num_ratings'] * item_stats['mean_rating'] + C * mu)
    / (item_stats['num_ratings'] + C)
)

item_stats.sort_values('bayes_score', ascending=False).head(10)

### 4.2 Top-N recommendation function

The function returns the highest-scoring eligible products after applying a minimum-review threshold. This threshold is explicit so that the recommendation list can balance reliability and catalog coverage.

In [ ]:
def recommend_top_n(item_stats, n=10, min_reviews=50):
    """
    Return the top-N items by Bayesian average score.

    Parameters
    ----------
    item_stats : pd.DataFrame
        DataFrame with columns: item_id, mean_rating, num_ratings, bayes_score
    n : int
        Number of items to return
    min_reviews : int
        Minimum number of ratings an item must have to be eligible

    Returns
    -------
    pd.DataFrame
        Top-N recommended items sorted by bayes_score descending
    """
    eligible = item_stats[item_stats['num_ratings'] >= min_reviews]
    return eligible.sort_values('bayes_score', ascending=False).head(n)


top_items = recommend_top_n(item_stats, n=10, min_reviews=50)
print(top_items)

In [ ]:
# Quick checks for different n and min_reviews values
print("Top 5, min_reviews=20")
print(recommend_top_n(item_stats, n=5, min_reviews=20).head(), "\n")

print("Top 10, min_reviews=100")
print(recommend_top_n(item_stats, n=10, min_reviews=100).head(), "\n")

print("Top 3, min_reviews=200")
print(recommend_top_n(item_stats, n=3, min_reviews=200))

## 5. Baseline evaluation strategy

A popularity model is a baseline, not a personalized solution. The chronological split below evaluates a simple global-mean predictor on later observations. It demonstrates the importance of measuring a model against future-like data rather than relying only on descriptive charts.

Further evaluation for a production recommender should include ranking metrics such as Precision@K, Recall@K, coverage, novelty, and diversity.

In [ ]:
# Baseline RMSE: predict global mean for every interaction

# Time-based split: first 80% train, last 20% test
df_sorted = df.sort_values('review_date').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.8)
train = df_sorted.iloc[:split_idx].copy()
test = df_sorted.iloc[split_idx:].copy()

# Compute RMSE on test set using train mean
mu_train = train['rating'].mean()
baseline_rmse = np.sqrt(np.mean((test['rating'] - mu_train) ** 2))
print(f"Baseline RMSE (predict global mean): {baseline_rmse:.4f}")

## 6. Interpretation and next steps

The analysis demonstrates why a transparent popularity baseline is valuable: it is simple to inspect, works for new users, and prevents low-volume products from dominating rankings because of a few favourable reviews. It is also a useful benchmark before investing in a more complex model.

Its limitation is that every user receives essentially the same ranking. It can overexpose mainstream items, underrepresent long-tail products, and react slowly to emerging preferences. A practical next step would be to compare this baseline with item-based collaborative filtering or a hybrid model that combines interaction data with product metadata.

### Reflection

The high sparsity in this dataset means personalized methods based on user similarity are hard to trust, because users rarely rate the same products. With so little overlap, neighbor relationships become noisy, and recommendations can be unstable unless we use stronger models or regularization.

A popularity-based recommender is weak when a user prefers specialized or uncommon products, since it mostly pushes mainstream items. It can also reduce recommendation diversity, overlook long-tail products, and respond slowly to new items or emerging interests.